# Study 886 — Agency MBS Carry — the teardown

The duration-neutral carry with empirical vs static-OAD hedges, the HAC *t* and block-bootstrap CI, the excess-vs-excess Sharpe race, the three-era cut, the HAC-lag sensitivity, the costed net, and the planted-carry synthetic control.

In [1]:
R = {'start': '2007-06-30', 'end': '2026-06-30', 'n_mbb': 229, 'n_vmbs': 199, 'fingerprint': '82b3eede7f92', 'mbb_beta': 0.521, 'mbb_r2': 0.69, 'mbb_carry': 0.3, 'mbb_t': 0.64, 'mbb_sharpe': 0.13, 'mbb_dd': -9.4, 'mbb_ci_lo': -0.62, 'mbb_ci_hi': 1.2, 'mbb_pneg': 0.26, 'mbb_static_carry': -0.3, 'mbb_static_t': -0.45, 'vmbs_beta': 0.544, 'vmbs_carry': 0.17, 'vmbs_t': 0.37, 'vmbs_static_carry': -0.15, 'race_mbs_sh': 0.336, 'race_mbs_mean': 1.41, 'race_mbs_vol': 4.2, 'race_ief_sh': 0.32, 'race_ief_mean': 2.14, 'race_ief_vol': 6.69, 'race_adv': 0.016, 'race_welch': -0.4, 'era1_carry': 1.83, 'era1_t': 2.5, 'era2_carry': 1.05, 'era2_t': 2.84, 'era3_carry': 0.17, 'era3_t': 0.18, 'hike_carry': 1.43, 'hike_t': 3.31, 'lag3_t': 0.61, 'lag6_t': 0.64, 'lag12_t': 0.64, 'net_mbb': -0.16, 'net_mbb_t': -0.33, 'charge': 0.45, 'net_vmbs': -0.3, 'y2009': 8.2, 'y2022': -4.4, 'syn_null_carry': 0.26, 'syn_null_t': 0.67, 'syn_planted_carry': 2.26, 'syn_planted_t': 5.95}

## The headline — duration-neutral carry `(MBS − cash) − β·(IEF − cash)`

`β` empirical = realized rate beta (~0.52, well below the static OAD ratio 0.80 — that gap is the negative-convexity signature).

In [2]:
print(f"MBB empirical  beta {R['mbb_beta']:.3f} (R2 {R['mbb_r2']:.2f}): "
      f"carry {R['mbb_carry']:+.2f}%/yr  HAC t = {R['mbb_t']:+.2f}  "
      f"Sharpe {R['mbb_sharpe']:+.2f}  maxDD {R['mbb_dd']:+.1f}%")
print(f"  bootstrap 95% CI [{R['mbb_ci_lo']:+.2f}, {R['mbb_ci_hi']:+.2f}] %/yr, "
      f"P(mean<0) = {R['mbb_pneg']:.2f}")
print(f"MBB static 0.80: carry {R['mbb_static_carry']:+.2f}%/yr  t = {R['mbb_static_t']:+.2f}  "
      f"-> NEGATIVE under a published-duration hedge")
print(f"VMBS empirical : carry {R['vmbs_carry']:+.2f}%/yr  t = {R['vmbs_t']:+.2f}  "
      f"(static {R['vmbs_static_carry']:+.2f}%/yr) -- corroborates")

MBB empirical  beta 0.521 (R2 0.69): carry +0.30%/yr  HAC t = +0.64  Sharpe +0.13  maxDD -9.4%
  bootstrap 95% CI [-0.62, +1.20] %/yr, P(mean<0) = 0.26
MBB static 0.80: carry -0.30%/yr  t = -0.45  -> NEGATIVE under a published-duration hedge
VMBS empirical : carry +0.17%/yr  t = +0.37  (static -0.15%/yr) -- corroborates


## Excess-vs-excess Sharpe race — MBS vs duration-matched IEF (both minus cash)

In [3]:
print(f"MBB excess: Sharpe {R['race_mbs_sh']:+.3f} ({R['race_mbs_mean']:+.2f}%/yr, "
      f"vol {R['race_mbs_vol']:.2f}%)")
print(f"IEF excess: Sharpe {R['race_ief_sh']:+.3f} ({R['race_ief_mean']:+.2f}%/yr, "
      f"vol {R['race_ief_vol']:.2f}%)")
print(f"-> Sharpe advantage {R['race_adv']:+.3f}  (raw Welch t = {R['race_welch']:+.2f}): a tie")

MBB excess: Sharpe +0.336 (+1.41%/yr, vol 4.20%)
IEF excess: Sharpe +0.320 (+2.14%/yr, vol 6.69%)
-> Sharpe advantage +0.016  (raw Welch t = -0.40): a tie


## Robustness — three eras (splits 2014-01, 2020-01)

The carry clears *t* >= 2 inside the calm sub-eras but dies across 2020-2026 — the very rate-vol regime the convexity premium exists to compensate for.

In [4]:
print(f"2007-2013 (GFC+recovery): {R['era1_carry']:+.2f}%/yr  HAC t = {R['era1_t']:+.2f}")
print(f"2014-2019 (QE grind)    : {R['era2_carry']:+.2f}%/yr  HAC t = {R['era2_t']:+.2f}")
print(f"2020-2026 (COVID+hiking): {R['era3_carry']:+.2f}%/yr  HAC t = {R['era3_t']:+.2f}  <- collapses")
print(f"  (2022+ hiking sub-window: {R['hike_carry']:+.2f}%/yr  t = {R['hike_t']:+.2f}, "
      f"but the full 2020-26 era it sits in is flat)")

2007-2013 (GFC+recovery): +1.83%/yr  HAC t = +2.50
2014-2019 (QE grind)    : +1.05%/yr  HAC t = +2.84
2020-2026 (COVID+hiking): +0.17%/yr  HAC t = +0.18  <- collapses
  (2022+ hiking sub-window: +1.43%/yr  t = +3.31, but the full 2020-26 era it sits in is flat)


## HAC-lag sensitivity — the thin full-sample *t* is not a lag artefact

In [5]:
for lags, t in [(3, R['lag3_t']), (6, R['lag6_t']), (12, R['lag12_t'])]:
    print(f"  NW lags={lags:>2d}: t = {t:+.2f}")

  NW lags= 3: t = +0.61
  NW lags= 6: t = +0.64
  NW lags=12: t = +0.64


## Tradability — costs push the thin carry below zero

ETF spreads (MBB 1 bp, IEF 2 bp one-way) on 12 rebalances/yr + 40 bps/yr borrow on the short Treasury leg.

In [6]:
print(f"MBB : gross {R['mbb_carry']:+.2f}%/yr - charge {R['charge']:.2f} "
      f"-> net {R['net_mbb']:+.2f}%/yr (HAC t = {R['net_mbb_t']:+.2f})")
print(f"VMBS: net {R['net_vmbs']:+.2f}%/yr")

MBB : gross +0.30%/yr - charge 0.45 -> net -0.16%/yr (HAC t = -0.33)
VMBS: net -0.30%/yr


## Synthetic positive control — the machinery is unbiased

Live: the estimator must NOT fire on the null and must recover a planted +2%/yr carry.

In [7]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from mbs_carry import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_world(carry_annual=0.0, seed=886+s))['t_hac'] for s in range(8)])
print(f"null (0 carry), 8 seeds: HAC t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_world(carry_annual=0.02, seed=886))
print(f"planted (+2%%/yr): recovered {planted['carry_ann_pct']:+.2f}%%/yr, HAC t = {planted['t_hac']:+.2f}, beta {planted['beta']:.3f}")

null (0 carry), 8 seeds: HAC t mean +0.49 (sd 0.69), |t|>=2 in 0/8
planted (+2%%/yr): recovered +2.26%%/yr, HAC t = +5.95, beta 0.525


## Verdict

- **Signal — Weak.** Right sign (MBB +0.30%/yr, VMBS +0.17%/yr) and *t* >= 2 inside the calm 2007-13 / 2014-19 sub-eras (*t* = +2.50 / +2.84), but full-sample HAC *t* = +0.64, bootstrap CI [-0.62, +1.20] straddles zero, the carry collapses to +0.17%/yr (*t* +0.18) in 2020-2026, flips negative (-0.30%/yr) under a static-OAD hedge, and the Sharpe advantage over IEF is +0.016. The synthetic control recovers a planted +2%/yr at *t* +5.95 and stays flat on the null (*t* +0.67), so this is a genuine *absence* of a robust premium, not machinery.
- **Tradability — Mirage.** The ~0.30%/yr gross carry is smaller than the ~0.45%/yr round-trip friction, so the costed net is **-0.16%/yr (MBB) / -0.30%/yr (VMBS)**; a proper static-duration hedge turns the spread negative before any cost. Negative convexity eats the option-adjusted spread.